In [ ]:
import sys
sys.path.append("..")

import os
import pandas as pd
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
from pprint import pp
import ipywidgets as widgets
from IPython.display import display, clear_output

from src.utils.visualize import mask_to_alpha_image

In [ ]:
output_dir = "../outputs"
run_groups = os.listdir(output_dir)
run_groups.sort()

def update_run_selector(change):
    runs = os.listdir(os.path.join(output_dir, change['new']))
    runs.sort()
    run_selector.options = runs
    run_selector.value = runs
    run_selector.layout.height = f"{8+len(runs)*20}px"

run_group_selector = widgets.ToggleButtons(options=run_groups, description="Select run group:")
run_group_selector.observe(update_run_selector, names="value")

runs = os.listdir(f"../outputs/{run_group_selector.value}")
runs.sort()
run_selector = widgets.SelectMultiple(
    options=runs,
    value=runs,
    description='Select runs:',
)
run_selector.layout.height = f"{8+len(runs)*19}px"

display(run_group_selector, run_selector)

In [ ]:
def flatten_config(conf):
    dict_ = OmegaConf.to_container(conf, resolve=True)
    return flatten_config_helper(dict_)

def flatten_config_helper(original, prefix=""):
    result = {}
    for key, val in original.items():
        if isinstance(val, dict):
            result = result | flatten_config_helper(val, prefix + key + ".")
        elif isinstance(val, list):
            if len(val)>0 and isinstance(val[0], dict): # flatten dicts to make hashable
                for dict_ in val:
                    result = result | flatten_config_helper(dict_, prefix + key + ".")
            else: # convert to tuple to make it hashable
                result[prefix + key] = tuple(val)
        else:
            result[prefix + key] = val
    return result

def config_diff(configs):
    sets = [set(conf.items()) for conf in configs]
    first_set = sets[0]
    set_diff = set()
    for set_ in sets[1:]:
        set_diff = set_diff.union(first_set ^ set_)
    return list(dict(set_diff).keys())

runs = run_selector.value
run_paths = [os.path.join(output_dir, run_group_selector.value, run) for run in runs]

config_list = [OmegaConf.load(os.path.join(path, ".hydra/config.yaml")) for path in run_paths]
for i, conf in enumerate(config_list):
    conf.run_name = runs[i]

if len(config_list) > 1:
    flattened_configs = [flatten_config(conf) for conf in config_list]
    config_diff_keys = config_diff(flattened_configs)
    conf_selector = widgets.SelectMultiple(
        options=config_diff_keys,
        value=config_diff_keys,
        description='Select config parameters to include:',
        )
    conf_selector.layout.height = f"{8+len(config_diff_keys)*19}px"
    display(conf_selector)

In [ ]:
if len(config_list) > 1:
    constant_config = {key: val for key, val in flattened_configs[0].items() if key not in config_diff_keys}
    selected_params = [{key: val for key, val in conf.items() if key in conf_selector.value} for conf in flattened_configs]
    non_selected_params = [{key: val for key, val in conf.items() if key in config_diff_keys and key not in conf_selector.value} for conf in flattened_configs]
    results_list = [pd.read_pickle(os.path.join(path, "results.pickle")) for path in run_paths]
    for i, results in enumerate(results_list):
        for name, val in selected_params[i].items():
            results[name] = val
    all_results = pd.concat(results_list)
    print("Shared config:")
    pp(constant_config)
    print("\nSelected parameters:")
    pp(selected_params)
    print("\nNon-selected parameters:")
    pp(non_selected_params)
else:
    all_results = pd.read_pickle(os.path.join(run_paths[0], "results.pickle"))
    print(OmegaConf.to_yaml(config_list[0]))

In [ ]:
#######################################################################
# Source - https://stackoverflow.com/a/34162576
# Posted by DSM
# Retrieved 2026-04-05, License - CC BY-SA 3.0

def query_df(df, dict_):
    return df.loc[(df[list(dict_)] == pd.Series(dict_)).all(axis=1)]

#######################################################################

all_results = all_results.fillna("nan")

available_metrics = [column_name[10:] for column_name in all_results.columns if column_name[:10] == "all_train_"]

options_dict = {key: all_results[key].unique() for key in conf_selector.value}

def plot_metric(query_dict, metric, log_scale):
    fig, ax = plt.subplots()
    query_result = query_df(all_results, query_dict)
    if len(query_result) > 1:
        print("Not unique combination!")
    elif len(query_result) == 0:
        print("No run satisfies combination!")
    else:
        ax.plot(query_result[f"all_train_{metric}"].item(), label=f"train_{metric}")
        ax.plot(query_result[f"all_test_{metric}"].item(), label=f"test_{metric}")
        if log_scale:
            ax.set_yscale("log")
        ax.legend()
        display(fig)

def plot_images(query_dict, cmap="summer"):
    query_result = query_df(all_results, query_dict)
    if len(query_result) > 1:
        print("Not unique combination!")
    elif len(query_result) == 0:
        print("No run satisfies combination!")
    else:
        fig, ax = plt.subplots(3, 2, figsize=(12,18))
        for i in range(3):
            ax[i, 0].imshow(query_result["target_image"].item(), cmap=cmap)
            ax[i, 0].set_title("target image")
        
        ax[0, 1].imshow(query_result["target_image"].item(), cmap=cmap)
        mask = query_result["train_mask"].item()
        ax[0, 1].imshow(mask_to_alpha_image(mask, red=0, green=0, blue=0))
        ax[0, 1].set_title("sparse (train)")
    
        ax[1, 1].imshow(query_result["reconstructed_target"].item(), cmap=cmap)
        ax[1, 1].set_title("reconstructed")
    
        ax[2, 1].imshow(query_result["target_image"].item(), cmap=cmap)
        mask = query_result["test_mask"].item()
        ax[2, 1].imshow(mask_to_alpha_image(mask, red=0, green=0, blue=0))
        ax[2, 1].set_title("test")
        for something in ax:
            for this in something:
                this.set_axis_off()
        display(fig)

def make_plots(change=None):
    with out:
        out.clear_output(wait=True)
        chosen_options = {list(options_dict)[i]: sel.value for i, sel in enumerate(options_selectors)}
        chosen_options["opencanopy_idx"] = opencanopy_idx_selector.value
        plot_metric(query_dict=chosen_options, metric=metric_selector.value, log_scale=log_scale_selector.value)
        plot_images(query_dict=chosen_options)


options_selectors = [widgets.ToggleButtons(options=vals, description=key+":") for key, vals in options_dict.items()]
opencanopy_idx_selector = widgets.Dropdown(
    options=all_results["opencanopy_idx"].unique(),
    description='opencanopy idx:',
)
metric_selector = widgets.Dropdown(
    options=available_metrics,
    value='loss',
    description='metric:',
)
log_scale_selector = widgets.Checkbox(
    value=False,
    description='log scale:',
)

for selector in options_selectors:
    selector.observe(make_plots, names="value")
metric_selector.observe(make_plots, names="value")
log_scale_selector.observe(make_plots, names="value")
opencanopy_idx_selector.observe(make_plots, names="value")


for selector in options_selectors:
    display(selector)
print("--------------------------------------------------------------------------------")
display(widgets.HBox([metric_selector, log_scale_selector]))
display(opencanopy_idx_selector)

out = widgets.Output()
out

In [ ]:
metric_columns = [f"train_{metric}" for metric in available_metrics] + [f"test_{metric}" for metric in available_metrics]
relevant_columns = list(conf_selector.value) + ["opencanopy_idx"] + metric_columns
final_metrics = all_results[relevant_columns]
final_metrics

In [ ]:
final_metrics.groupby(["data.sparsity", "model.scale"]).mean().sort_values(["data.sparsity", "test_loss"], ascending=True)

In [ ]:
# let us assume all combinations of chosen parameters are present
def make_image_grid(opencanopy_idx, x_var, y_var, flip_x, flip_y, image_name, mask_name, metric, log_scale, color_within):
    assert (image_name is None and metric is not None) or (image_name is not None and metric is None)
    size = 4
    x_vals = all_results[x_var].sort_values(ascending=flip_x).unique()
    y_vals = all_results[y_var].sort_values(ascending=flip_y).unique()
    x_len = len(x_vals)
    y_len = len(y_vals)
    fig, ax = plt.subplots(y_len, x_len, figsize=(size*x_len, size*y_len))
    for i, y_val in enumerate(y_vals):
        if color_within == y_var:
            min_loss = all_results.query(f"opencanopy_idx == {opencanopy_idx} and `{y_var}` == {y_val}")[f"test_loss"].min()
            max_loss = all_results.query(f"opencanopy_idx == {opencanopy_idx} and `{y_var}` == {y_val}")[f"test_loss"].max()
        for j, x_val in enumerate(x_vals):
            if color_within == x_var:
                min_loss = all_results.query(f"opencanopy_idx == {opencanopy_idx} and `{x_var}` == {x_val}")[f"test_loss"].min()
                max_loss = all_results.query(f"opencanopy_idx == {opencanopy_idx} and `{x_var}` == {x_val}")[f"test_loss"].max()
            if metric:
                ax[i, j].plot(all_results.query(f"opencanopy_idx == {opencanopy_idx} and `{y_var}` == {y_val} and `{x_var}` == {x_val}")[f"all_train_{metric}"].item(), label=f"train_{metric}")
                ax[i, j].plot(all_results.query(f"opencanopy_idx == {opencanopy_idx} and `{y_var}` == {y_val} and `{x_var}` == {x_val}")[f"all_test_{metric}"].item(), label=f"test_{metric}")
                if log_scale:
                    ax[i, j].set_yscale("log")
                ax[i, j].legend()
            else:
                ax[i, j].imshow(all_results.query(f"opencanopy_idx == {opencanopy_idx} and `{y_var}` == {y_val} and `{x_var}` == {x_val}")[image_name].item(), cmap="summer")
                if mask_name is not None:
                    mask = all_results.query(f"opencanopy_idx == {opencanopy_idx} and `{y_var}` == {y_val} and `{x_var}` == {x_val}")[mask_name].item()
                    ax[i, j].imshow(mask_to_alpha_image(mask, red=0, green=0, blue=0))
                ax[i, j].set_xticks([], [])
                ax[i, j].set_yticks([], [])
                loss_value = all_results.query(f"opencanopy_idx == {opencanopy_idx} and `{y_var}` == {y_val} and `{x_var}` == {x_val}")[f"test_loss"].item()
                ax[i, j].patch.set_edgecolor((float((max_loss-loss_value)/(max_loss-min_loss))**1, 0.0, 0.0))
                ax[i, j].patch.set_linewidth(8)
            ax[i, j].set_title(f"{y_var}={y_val}, {x_var}={x_val}")
    fig.tight_layout()
    display(fig)

def update_image_grid(update):
    match image_selector.value:
        case "target":
            image_column = "target_image"
            mask_column = None
            metric = None
        case "sparse":
            image_column = "target_image"
            mask_column = "train_mask"
            metric = None
        case "reconstructed":
            image_column = "reconstructed_target"
            mask_column = None
            metric = None
        case "test_grid":
            image_column = "target_image"
            mask_column = "test_mask"
            metric = None
        case "metric":
            image_column = None
            mask_column = None
            metric = metric_selector_grid.value
    with out2:
        out2.clear_output(wait=True)
        make_image_grid(opencanopy_idx_selector_grid.value, x_var_selector.value, y_var_selector.value, x_flip.value, y_flip.value, image_column, mask_column, metric, log_scale_selector_grid.value, color_within_selector.value)

opencanopy_idx_selector_grid = widgets.ToggleButtons(
    options=all_results["opencanopy_idx"].unique(),
    description='opencanopy idx:',
)
image_selector = widgets.ToggleButtons(
    options=["target", "sparse", "reconstructed", "test_grid", "metric"],
    description="show:"
)
metric_selector_grid = widgets.Dropdown(
    options=available_metrics,
    description="metric:"
)
log_scale_selector_grid = widgets.Checkbox(
    description="log scale"
)
x_var_selector = widgets.ToggleButtons(
    options=list(conf_selector.value),
    description="x_var:"
)
y_var_selector = widgets.ToggleButtons(
    options=list(conf_selector.value),
    description="y_var:"
)
x_flip = widgets.Checkbox(
    description="flip x-axis:"
)
y_flip = widgets.Checkbox(
    description="flip y-axis:"
)
color_within_selector = widgets.ToggleButtons(
    options=list(conf_selector.value),
    description="color within:"
)

opencanopy_idx_selector_grid.observe(update_image_grid, names="value")
image_selector.observe(update_image_grid, names="value")
x_var_selector.observe(update_image_grid, names="value")
y_var_selector.observe(update_image_grid, names="value")
x_flip.observe(update_image_grid, names="value")
y_flip.observe(update_image_grid, names="value")
metric_selector_grid.observe(update_image_grid, names="value")
log_scale_selector_grid.observe(update_image_grid, names="value")
color_within_selector.observe(update_image_grid, names="value")

display(opencanopy_idx_selector_grid)
display(widgets.HBox([x_var_selector, x_flip]))
display(widgets.HBox([y_var_selector, y_flip]))
display(widgets.HBox([image_selector, metric_selector_grid, log_scale_selector_grid]))
display(color_within_selector)

out2 = widgets.Output()
out2

In [ ]:
# Beware
# Either give range or provide colorbar for images. Maybe its good to give range either ways and maybe also for metrics.